# Expérimentation des modèles ML — 9 catégories Oscar

> **Pendant empirique** du notebook [`Model justification.ipynb`](Model%20justification.ipynb).
> On y a posé les hypothèses (5 modèles candidats × 9 catégories, top-pick théorique, feature engineering ciblé).
> Ici on **mesure** : quel modèle gagne réellement par catégorie, sur quelle métrique métier ?

**Pipeline** :
1. Feature engineering générique (§4 de la justification) → film + historiques personne + cross-catégorie + texte.
2. Évaluation `GroupKFold(groups=year)` × 5 folds, métriques : **top-1 accuracy par année** (métier), PR-AUC, log-loss.
3. 5 modèles candidats par catégorie (LR L2, Random Forest, AdaBoost, XGBoost, LightGBM) + Stacking sur les catégories où la justification le préconise.
4. Synthèse : tableau croisé `(catégorie × modèle) → top-1 acc ± std`, comparaison avec les paris théoriques.

**Anti-leakage** (rappels critiques de la justification §15) :
- Features historiques personne calculées avec `cumcount()` + `shift(1)` ⇒ jamais d'info future.
- Features cross-catégorie (`film_n_total_noms`) calculées sur `tconst × year` sans utiliser `winner`.
- Split CV par année : `GroupKFold(groups=df.year)` — jamais de split aléatoire (une année doit être *entièrement* dans un fold).
- TF-IDF refitté *par fold* via `Pipeline` sklearn.
- `class_weight='balanced'` (ou `scale_pos_weight`) ; **pas de SMOTE** (échantillons trop petits).

**Auteurs** : Anna / Robin / Jonathan — *suite directe du notebook de Keira*.

## 0. Setup

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import (
    AdaBoostClassifier,
    RandomForestClassifier,
    StackingClassifier,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, log_loss
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
RNG = 42

# Chemin du dataset final (cf. CLAUDE.md)
DATA_PATH = Path('../Data/Processed/oscar_imdb_merged.parquet')
assert DATA_PATH.exists(), f'Dataset introuvable : {DATA_PATH.resolve()}'

# 9 catégories cibles (justification §0.1)
TARGETS: dict[str, str] = {
    'Best Actor in a Leading Role':        'Meilleur Acteur',
    'Best Actress in a Leading Role':      'Meilleure Actrice',
    'Best Actor in a Supporting Role':     'Meilleur Acteur Second Rôle',
    'Best Actress in a Supporting Role':   'Meilleure Actrice Second Rôle',
    'Best Picture':                        'Meilleur Film',
    'Best Directing':                      'Meilleur Réalisateur',
    'Best Animated Feature Film':          'Meilleur Film Animé',
    'Best Visual Effects':                 'Meilleurs Effets Visuels',
    'Best Writing (Original Screenplay)':  'Meilleur Scénario Original',
}

N_SPLITS = 5  # GroupKFold(n_splits=5)

## 1. Chargement et inspection rapide

In [ ]:
df_raw = pd.read_parquet(DATA_PATH)
print(f'Dataset : {df_raw.shape[0]} lignes × {df_raw.shape[1]} colonnes — années {df_raw.year.min()}–{df_raw.year.max()}')

rows = []
for cat_en, cat_fr in TARGETS.items():
    s = df_raw[df_raw.category == cat_en]
    rows.append((cat_fr, cat_en, len(s), int(s.winner.sum()), s.winner.mean()))
summary = pd.DataFrame(rows, columns=['Catégorie (FR)', 'category (dataset)', 'n', 'winners', 'base_rate'])
summary['base_rate'] = summary['base_rate'].map('{:.1%}'.format)
summary

## 2. Helpers — parsing & utilitaires (principe DRY)

Les colonnes `genres`, `keywords`, `production_countries` arrivent sous forme variable (string CSV, `np.ndarray`, `None`). On les normalise une fois pour toutes vers `list[str]`.

In [ ]:
def to_list(value) -> list[str]:
    """Normalise une valeur (str CSV / ndarray / list / None) vers list[str]."""
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []
    if isinstance(value, (list, tuple)):
        return [str(x) for x in value if x is not None]
    if isinstance(value, np.ndarray):
        return [str(x) for x in value.tolist() if x is not None]
    if isinstance(value, str):
        return [s.strip() for s in value.split(',') if s.strip()]
    return []


def safe_log1p(s: pd.Series) -> pd.Series:
    """`log1p` qui transforme les 0 en NaN (un revenue=0 = 'inconnu', pas 'gratuit')."""
    return np.log1p(s.where(s > 0))


def parse_month(date_str) -> float:
    if not isinstance(date_str, str) or len(date_str) < 7:
        return np.nan
    try:
        return float(date_str[5:7])
    except Exception:
        return np.nan

## 3. Feature engineering — film (générique, §4.1 de la justification)

Features applicables à toutes les catégories. Numériques, binaires, et bins ordinaux.

In [ ]:
TOP_GENRES = [
    'Drama', 'Comedy', 'Action', 'Thriller', 'Adventure',
    'Crime', 'Biography', 'Romance', 'Mystery', 'Sci-Fi',
    'Fantasy', 'History', 'War', 'Animation', 'Music',
]

VFX_GENRES = {'Sci-Fi', 'Fantasy', 'Action', 'Adventure'}


def build_film_features(df: pd.DataFrame) -> pd.DataFrame:
    """Construit les features 'film' identiques pour toutes les catégories.

    Renvoie un DataFrame aligné sur df.index.
    """
    out = pd.DataFrame(index=df.index)

    # Numériques transformées
    out['log_imdb_votes'] = safe_log1p(df['imdb_votes'])
    out['log_budget'] = safe_log1p(df['budget'])
    out['log_revenue'] = safe_log1p(df['revenue'])
    out['imdb_rating'] = df['imdb_rating']
    out['tmdb_vote_average'] = df['tmdb_vote_average']
    out['log_tmdb_vote_count'] = safe_log1p(df['tmdb_vote_count'])
    out['rating_gap'] = df['imdb_rating'] - df['tmdb_vote_average']
    out['runtime_minutes'] = df['runtime_minutes'].astype(float)
    out['n_genres'] = df['n_genres'].astype(float)
    out['n_cast'] = df['n_cast'].astype(float)

    # ROI (cap à 50 pour neutraliser les explosions)
    roi = df['revenue'] / df['budget'].replace(0, np.nan)
    out['roi'] = roi.clip(0, 50)

    # Timing de sortie
    month = df['release_date'].apply(parse_month)
    out['release_month'] = month
    out['release_quarter'] = ((month - 1) // 3 + 1).astype(float)
    out['is_late_release'] = (month >= 10).astype(float)

    # Bins de runtime (ordinal)
    out['runtime_bin'] = pd.cut(
        df['runtime_minutes'].astype(float),
        bins=[0, 90, 120, 150, 1000],
        labels=[0, 1, 2, 3],
    ).astype(float)
    out['runtime_long'] = (df['runtime_minutes'].astype(float) > 150).astype(float)

    # One-hot top genres (cardinalité faible, OK)
    genres_lists = df['genres'].apply(to_list)
    for g in TOP_GENRES:
        out[f'genre_{g.lower()}'] = genres_lists.apply(lambda lst, g=g: float(g in lst))
    out['genre_is_biopic'] = genres_lists.apply(
        lambda lst: float(('Biography' in lst) or ('History' in lst))
    )
    out['genre_is_vfx_heavy'] = genres_lists.apply(
        lambda lst: float(any(g in VFX_GENRES for g in lst))
    )
    out['genre_is_war_or_historical'] = genres_lists.apply(
        lambda lst: float(('War' in lst) or ('History' in lst))
    )
    out['genre_is_drama_or_comedy'] = genres_lists.apply(
        lambda lst: float(('Drama' in lst) or ('Comedy' in lst))
    )
    out['genre_is_drama_or_crime'] = genres_lists.apply(
        lambda lst: float(('Drama' in lst) or ('Crime' in lst))
    )

    # Pays / langue
    countries_lists = df['production_countries'].apply(to_list)
    out['is_english'] = (df['original_language'] == 'en').astype(float)
    out['is_us_prod'] = countries_lists.apply(lambda lst: float('US' in lst))
    out['n_countries'] = countries_lists.apply(len).astype(float)
    out['is_coproduction'] = (out['n_countries'] > 1).astype(float)

    # Texte (longueurs uniquement ici, TF-IDF en pipeline)
    out['tagline_len'] = df['tagline'].fillna('').str.split().apply(len).astype(float)
    out['overview_len'] = df['overview'].fillna('').str.split().apply(len).astype(float)
    keywords_lists = df['keywords'].apply(to_list)
    out['n_keywords'] = keywords_lists.apply(len).astype(float)

    # Décennie
    out['decade'] = (df['year'] // 10 * 10).astype(float)

    # Budget z-scoré par décennie (§4.1 binning + normalisation)
    by_dec = df.assign(_lb=out['log_budget'], _dec=out['decade']).groupby('_dec')['_lb']
    out['log_budget_z_decade'] = (out['log_budget'] - by_dec.transform('mean')) / by_dec.transform('std')

    # Budget bins (Best Picture & VFX)
    out['budget_bin'] = pd.cut(
        df['budget'].replace(0, np.nan),
        bins=[0, 1e7, 5e7, 1e8, 1e10],
        labels=[0, 1, 2, 3],
    ).astype(float)

    return out


film_feat = build_film_features(df_raw)
print(f'Features film : {film_feat.shape[1]} colonnes (sur {len(df_raw)} lignes)')
film_feat.head(3)

## 4. Feature engineering — historique personne (§4.2)

**Anti-leakage critique** : pour chaque `nconst`, on calcule `n_prior_noms` et `n_prior_wins` *strictement avant* l'année courante.

Méthode robuste : agréger d'abord à la maille `(nconst, year)` (un acteur peut être nominé dans 2 catégories la même année → ça compte pour **1** nomination cumulée historique, pas 2), puis `cumcount` + `cumsum` triés par année, et soustraire la ligne courante.

In [ ]:
def build_person_history(df: pd.DataFrame) -> pd.DataFrame:
    """Pour chaque (nconst, year), calcule l'historique cumulé strictement antérieur.

    Renvoie un DataFrame indexé par (nconst, year) avec :
      - n_prior_noms : nb de cérémonies passées avec ≥ 1 nomination
      - n_prior_wins : nb de victoires passées
      - years_since_last_nom / years_since_last_win
    """
    sub = df.dropna(subset=['nconst']).copy()
    agg = (sub.groupby(['nconst', 'year'], as_index=False)
              .agg(any_winner=('winner', 'max')))
    agg = agg.sort_values(['nconst', 'year']).reset_index(drop=True)

    # cumcount = nb de lignes précédentes pour ce nconst (= nb de cérémonies passées)
    agg['n_prior_noms'] = agg.groupby('nconst').cumcount()

    # cumsum des wins puis on retire la ligne courante
    cum_wins = agg.groupby('nconst')['any_winner'].cumsum()
    agg['n_prior_wins'] = (cum_wins - agg['any_winner'].astype(int)).astype(int)

    # années depuis la dernière nomination / dernière victoire
    agg['_last_year'] = agg.groupby('nconst')['year'].shift(1)
    agg['years_since_last_nom'] = agg['year'] - agg['_last_year']

    # années depuis dernière victoire : ne tenir compte que des wins passées
    agg['_win_year'] = np.where(agg['any_winner'], agg['year'], np.nan)
    agg['_last_win_year'] = (agg.groupby('nconst')['_win_year']
                                .apply(lambda s: s.ffill().shift(1))
                                .reset_index(level=0, drop=True))
    agg['years_since_last_win'] = agg['year'] - agg['_last_win_year']

    agg = agg.drop(columns=['_last_year', '_win_year', '_last_win_year', 'any_winner'])
    return agg.set_index(['nconst', 'year'])


person_hist = build_person_history(df_raw)
print(f'Historique personne : {len(person_hist)} (nconst, year) uniques')
person_hist.head(5)

## 5. Feature engineering — cross-catégorie (§9.5)

Pour un film donné à une cérémonie donnée :
- `film_n_total_noms` : nb total de catégories où le film est nominé (signal reine pour Best Picture).
- `film_is_BP_nominee`, `has_director_nom`, `has_acting_nom` : drapeaux de buzz.

**Anti-leakage** : ces features se calculent sur le dataset complet **sans utiliser `winner`** → pas de fuite ; on n'utilise que la présence de nominations.

Garde-fou §15.2 : pour la catégorie Best Picture, `film_is_BP_nominee` vaudrait toujours `True` → on utilise `n_other_noms` (nominations dans les autres catégories) à la place.

In [ ]:
ACTING_CATS = {
    'Best Actor in a Leading Role',
    'Best Actress in a Leading Role',
    'Best Actor in a Supporting Role',
    'Best Actress in a Supporting Role',
}


def build_cross_category_features(df: pd.DataFrame) -> pd.DataFrame:
    """Pour chaque ligne, calcule des compteurs au niveau (tconst, year)."""
    sub = df.dropna(subset=['tconst']).copy()

    grp = sub.groupby(['tconst', 'year'])
    cnt = grp['category'].nunique().rename('film_n_total_noms')
    is_bp = (grp['category'].apply(lambda s: 'Best Picture' in set(s))
                .astype(float).rename('film_is_BP_nominee'))
    has_dir = (grp['category'].apply(lambda s: 'Best Directing' in set(s))
                  .astype(float).rename('has_director_nom'))
    has_act = (grp['category'].apply(lambda s: bool(set(s) & ACTING_CATS))
                  .astype(float).rename('has_acting_nom'))
    has_writ = (grp['category'].apply(
                  lambda s: bool(set(s) & {'Best Writing (Original Screenplay)',
                                            'Best Writing (Adapted Screenplay)'}))
                  .astype(float).rename('has_writing_nom'))

    feats = pd.concat([cnt, is_bp, has_dir, has_act, has_writ], axis=1)
    # Merge sur (tconst, year) — les films sans tconst gardent NaN
    out = df[['tconst', 'year']].merge(
        feats.reset_index(), on=['tconst', 'year'], how='left'
    )
    out.index = df.index
    return out.drop(columns=['tconst', 'year'])


cross_feat = build_cross_category_features(df_raw)
print(f'Features cross-catégorie : {cross_feat.shape}')
cross_feat.describe().round(2)

## 6. Assemblage de la matrice par catégorie

Stratégie : on précalcule tous les blocs de features une seule fois (film + cross-cat + historique personne) sur `df_raw`, puis pour chaque catégorie cible on extrait les lignes pertinentes et on **ajoute les features spécifiques** (`age_bin`, `is_overdue`, `breakthrough_in_indie`, etc.) selon la justification §5–13.

Cohérence DRY : un seul builder `build_X(category)` orchestre tout.

In [ ]:
# Précalcul global
FILM_FEATS = build_film_features(df_raw)
CROSS_FEATS = build_cross_category_features(df_raw)
PERSON_HIST = build_person_history(df_raw)


def _attach_person_history(df_cat: pd.DataFrame) -> pd.DataFrame:
    """Joint l'historique personne à df_cat via (nconst, year)."""
    keys = list(zip(df_cat['nconst'], df_cat['year']))
    hist = PERSON_HIST.reindex(keys)
    hist.index = df_cat.index
    # Pour les lignes sans nconst (catégories 'film'), valeurs NaN → on remplit à 0
    return hist.fillna({'n_prior_noms': 0, 'n_prior_wins': 0})


def _build_text_combined(df_cat: pd.DataFrame) -> pd.Series:
    """Concatène keywords + overview pour TF-IDF."""
    kw = df_cat['keywords'].apply(lambda x: ' '.join(to_list(x)))
    ov = df_cat['overview'].fillna('').astype(str)
    return (kw + ' ' + ov).str.strip()


def build_X_for_category(category: str) -> tuple[pd.DataFrame, np.ndarray, np.ndarray, pd.DataFrame]:
    """Renvoie (X, y, groups, df_cat) pour la catégorie demandée.

    X est un DataFrame mixte (numérique + colonne `text_combined` quand utile).
    groups = year (pour GroupKFold).
    df_cat = sous-DataFrame brut (utile pour calculer le top-1 par année).
    """
    mask = df_raw['category'] == category
    df_cat = df_raw[mask].copy()

    parts = [FILM_FEATS.loc[mask].copy(), CROSS_FEATS.loc[mask].copy()]
    hist = _attach_person_history(df_cat)
    parts.append(hist)

    X = pd.concat(parts, axis=1)

    # --- Features spécifiques par catégorie (issues de §5–13) ---
    X['is_first_nomination'] = (hist['n_prior_noms'].fillna(0) == 0).astype(float)
    X['is_overdue'] = ((hist['n_prior_noms'].fillna(0) >= 3) &
                       (hist['n_prior_wins'].fillna(0) == 0)).astype(float)
    X['is_veteran'] = (hist['n_prior_noms'].fillna(0) >= 2).astype(float)

    # is_indie : budget dans le tercile bas de l'année
    df_cat_yr = df_cat.assign(_b=df_cat['budget'].replace(0, np.nan))
    q33 = df_cat_yr.groupby('year')['_b'].transform(lambda s: s.quantile(0.33))
    X['is_indie'] = (df_cat_yr['_b'] < q33).astype(float).fillna(0.0)

    # Pour Best Picture : remplacer film_is_BP_nominee (toujours True) par n_other_noms
    if category == 'Best Picture':
        X['n_other_noms'] = (X['film_n_total_noms'] - 1).clip(lower=0)
        X = X.drop(columns=['film_is_BP_nominee'])

    # Best Directing : remplacer has_director_nom (toujours True) par d'autres signaux
    if category == 'Best Directing':
        X = X.drop(columns=['has_director_nom'])

    # Acteurs second rôle : interactions explicites
    if category == 'Best Actor in a Supporting Role':
        X['veteran_in_BP'] = X['is_veteran'] * X['film_is_BP_nominee']
        X['breakthrough_in_BP'] = X['is_first_nomination'] * X['film_is_BP_nominee']
    if category == 'Best Actress in a Supporting Role':
        X['breakthrough_in_indie'] = X['is_first_nomination'] * X['is_indie']

    # Texte (catégories où le contenu narratif compte)
    text_cats = {'Best Picture', 'Best Writing (Original Screenplay)', 'Best Animated Feature Film'}
    if category in text_cats:
        X['text_combined'] = _build_text_combined(df_cat)

    # On ne garde que les lignes avec un tconst (sinon impossible à utiliser)
    valid = df_cat['tconst'].notna()
    X = X.loc[valid]
    df_cat = df_cat.loc[valid]

    y = df_cat['winner'].astype(int).values
    groups = df_cat['year'].values
    return X, y, groups, df_cat


# Sanity-check sur Best Picture
X_bp, y_bp, g_bp, df_bp = build_X_for_category('Best Picture')
print(f'Best Picture — X: {X_bp.shape}, winners: {y_bp.sum()}, années: {sorted(set(g_bp))[:5]}...{sorted(set(g_bp))[-3:]}')
X_bp.head(3)

## 7. Framework d'évaluation

Métrique métier : **top-1 accuracy par année** = `1[argmax(P) == winner]` agrégé sur les `(category, year)` du fold de test.

Pour chaque fold, on entraîne le modèle, prédit `proba` sur le test, et pour chaque année du test on vérifie que le nominé avec la plus forte proba est bien le winner. La métrique finale = moyenne sur toutes les années des 5 folds.

On rapporte aussi PR-AUC (déséquilibre) et log-loss (calibration).

In [ ]:
def evaluate_by_group(
    X: pd.DataFrame,
    y: np.ndarray,
    groups: np.ndarray,
    model_factory,
    n_splits: int = N_SPLITS,
) -> dict:
    """Évalue un modèle en GroupKFold(groups=year).

    model_factory : callable() → estimator non-fitté (pour repartir à zéro à chaque fold).
    Renvoie un dict avec les moyennes et écarts-type.
    """
    gkf = GroupKFold(n_splits=n_splits)
    top1_per_year = []
    ap_per_fold = []
    ll_per_fold = []

    for tr, te in gkf.split(X, y, groups=groups):
        model = model_factory()
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y[tr], y[te]

        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_te)[:, 1]

        # Top-1 par année sur le fold
        df_te = pd.DataFrame({
            'year': groups[te],
            'winner': y_te,
            'proba': proba,
        })
        for yr, g in df_te.groupby('year'):
            if g['winner'].sum() == 0:
                continue  # année sans gagnant dans le sous-ensemble (cas limite)
            pred_idx = g['proba'].idxmax()
            top1_per_year.append(int(g.loc[pred_idx, 'winner'] == 1))

        # PR-AUC + log-loss
        if y_te.sum() > 0 and y_te.sum() < len(y_te):
            ap_per_fold.append(average_precision_score(y_te, proba))
        proba_c = np.clip(proba, 1e-6, 1 - 1e-6)
        ll_per_fold.append(log_loss(y_te, proba_c, labels=[0, 1]))

    return {
        'top1_acc': float(np.mean(top1_per_year)) if top1_per_year else float('nan'),
        'top1_std': float(np.std(top1_per_year)) if top1_per_year else float('nan'),
        'pr_auc':   float(np.mean(ap_per_fold)) if ap_per_fold else float('nan'),
        'log_loss': float(np.mean(ll_per_fold)) if ll_per_fold else float('nan'),
        'n_years_evaluated': len(top1_per_year),
    }

## 8. Définition des modèles candidats

Chaque modèle est une **factory** (`callable() → Pipeline`) pour repartir d'un estimateur frais à chaque fold (TF-IDF/SVD ré-entraînés à l'intérieur du pipeline).

On utilise `ColumnTransformer` pour router les colonnes :
- numériques → `SimpleImputer(median) + StandardScaler`
- `text_combined` (str) → `TfidfVectorizer + TruncatedSVD(20)`

Régularisation forte partout (justification §3) : grilles fixées (pas de GridSearchCV pour ne pas surinflater la grille — c'est une *première passe*).

In [ ]:
def _split_columns(X: pd.DataFrame) -> tuple[list[str], list[str]]:
    text_cols = [c for c in X.columns if c == 'text_combined']
    num_cols = [c for c in X.columns if c not in text_cols]
    return num_cols, text_cols


def _make_preprocessor(X: pd.DataFrame, scale: bool = True) -> ColumnTransformer:
    """ColumnTransformer adapté à X : num → impute+scale, text → TF-IDF + SVD."""
    num_cols, text_cols = _split_columns(X)
    num_steps = [('imp', SimpleImputer(strategy='median'))]
    if scale:
        num_steps.append(('sc', StandardScaler()))
    transformers = [('num', Pipeline(num_steps), num_cols)]
    if text_cols:
        text_pipe = Pipeline([
            ('tfidf', TfidfVectorizer(max_features=300, ngram_range=(1, 2),
                                       min_df=2, max_df=0.9)),
            ('svd', TruncatedSVD(n_components=20, random_state=RNG)),
        ])
        transformers.append(('text', text_pipe, 'text_combined'))
    return ColumnTransformer(transformers, remainder='drop')


def make_lr_l2(X: pd.DataFrame):
    return Pipeline([
        ('prep', _make_preprocessor(X, scale=True)),
        ('clf', LogisticRegression(C=1.0, penalty='l2', class_weight='balanced',
                                    max_iter=2000, random_state=RNG)),
    ])


def make_lr_enet(X: pd.DataFrame):
    return Pipeline([
        ('prep', _make_preprocessor(X, scale=True)),
        ('clf', LogisticRegression(C=1.0, penalty='elasticnet', l1_ratio=0.5,
                                    solver='saga', class_weight='balanced',
                                    max_iter=4000, random_state=RNG)),
    ])


def make_decision_tree(X: pd.DataFrame, max_depth: int = 4):
    return Pipeline([
        ('prep', _make_preprocessor(X, scale=False)),
        ('clf', DecisionTreeClassifier(max_depth=max_depth, min_samples_leaf=5,
                                        class_weight='balanced', random_state=RNG)),
    ])


def make_random_forest(X: pd.DataFrame, max_depth: int = 5):
    return Pipeline([
        ('prep', _make_preprocessor(X, scale=False)),
        ('clf', RandomForestClassifier(n_estimators=300, max_depth=max_depth,
                                        min_samples_leaf=5, class_weight='balanced',
                                        n_jobs=-1, random_state=RNG)),
    ])


def make_adaboost(X: pd.DataFrame):
    base = DecisionTreeClassifier(max_depth=2, random_state=RNG)
    return Pipeline([
        ('prep', _make_preprocessor(X, scale=False)),
        ('clf', AdaBoostClassifier(estimator=base, n_estimators=100,
                                    learning_rate=0.5, random_state=RNG)),
    ])


def make_xgboost(X: pd.DataFrame, max_depth: int = 4, lr: float = 0.05, n_estimators: int = 400):
    return Pipeline([
        ('prep', _make_preprocessor(X, scale=False)),
        ('clf', xgb.XGBClassifier(
            max_depth=max_depth, learning_rate=lr, n_estimators=n_estimators,
            subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            random_state=RNG, eval_metric='logloss', tree_method='hist',
            verbosity=0,
        )),
    ])


def make_lightgbm(X: pd.DataFrame, num_leaves: int = 15):
    return Pipeline([
        ('prep', _make_preprocessor(X, scale=False)),
        ('clf', lgb.LGBMClassifier(
            num_leaves=num_leaves, min_data_in_leaf=10,
            learning_rate=0.05, n_estimators=400,
            feature_fraction=0.8, class_weight='balanced',
            random_state=RNG, verbosity=-1,
        )),
    ])


def make_stacking_lr_xgb(X: pd.DataFrame):
    """Stacking LR + XGBoost → LR meta. Pour Best Picture / Best Screenplay."""
    pre = _make_preprocessor(X, scale=True)
    base = [
        ('lr', LogisticRegression(C=1.0, class_weight='balanced',
                                  max_iter=2000, random_state=RNG)),
        ('xgb', xgb.XGBClassifier(
            max_depth=4, learning_rate=0.05, n_estimators=300,
            subsample=0.9, colsample_bytree=0.9,
            random_state=RNG, eval_metric='logloss', tree_method='hist',
            verbosity=0,
        )),
    ]
    stack = StackingClassifier(
        estimators=base,
        final_estimator=LogisticRegression(C=1.0, max_iter=2000, random_state=RNG),
        cv=3, n_jobs=1, passthrough=False,
    )
    return Pipeline([('prep', pre), ('clf', stack)])

### 8.1 Mapping catégorie → liste des modèles à comparer

Pour chaque catégorie, les 5 modèles candidats listés dans la justification (§5.3 → §13.3). Ordre du tableau §14 respecté.

In [ ]:
MODEL_PLAN: dict[str, list[str]] = {
    'Best Actor in a Leading Role':        ['lr_l2', 'random_forest', 'xgboost', 'lightgbm', 'stacking_lr_xgb'],
    'Best Actress in a Leading Role':      ['lr_l2', 'lr_enet', 'random_forest', 'xgboost', 'stacking_lr_xgb'],
    'Best Actor in a Supporting Role':     ['lr_l2', 'random_forest', 'adaboost', 'xgboost', 'stacking_lr_xgb'],
    'Best Actress in a Supporting Role':   ['lr_l2', 'decision_tree', 'random_forest', 'xgboost', 'lightgbm'],
    'Best Picture':                        ['lr_l2', 'random_forest', 'xgboost', 'lightgbm', 'stacking_lr_xgb'],
    'Best Directing':                      ['lr_l2', 'random_forest', 'xgboost', 'lightgbm', 'stacking_lr_xgb'],
    'Best Animated Feature Film':          ['decision_tree', 'random_forest', 'lr_l2', 'xgboost', 'lightgbm'],
    'Best Visual Effects':                 ['lr_l2', 'decision_tree', 'random_forest', 'xgboost', 'lightgbm'],
    'Best Writing (Original Screenplay)':  ['lr_l2', 'random_forest', 'xgboost', 'lightgbm', 'stacking_lr_xgb'],
}

MODEL_FACTORIES = {
    'lr_l2':           make_lr_l2,
    'lr_enet':         make_lr_enet,
    'decision_tree':   make_decision_tree,
    'random_forest':   make_random_forest,
    'adaboost':        make_adaboost,
    'xgboost':         make_xgboost,
    'lightgbm':        make_lightgbm,
    'stacking_lr_xgb': make_stacking_lr_xgb,
}

# Top-pick théorique (§14) — pour comparaison ex-post
THEORETICAL_PICK = {
    'Best Actor in a Leading Role':        'lr_l2',
    'Best Actress in a Leading Role':      'lr_enet',
    'Best Actor in a Supporting Role':     'random_forest',
    'Best Actress in a Supporting Role':   'lr_l2',
    'Best Picture':                        'xgboost',
    'Best Directing':                      'lightgbm',
    'Best Animated Feature Film':          'random_forest',
    'Best Visual Effects':                 'xgboost',
    'Best Writing (Original Screenplay)':  'stacking_lr_xgb',
}

## 9. Boucle d'expérimentation

Pour chaque catégorie cible : on construit X/y/groups, on évalue les 5 modèles en GroupKFold, on stocke les résultats.

In [ ]:
results = []

for cat_en, cat_fr in TARGETS.items():
    X_cat, y_cat, g_cat, df_cat = build_X_for_category(cat_en)
    base_rate = y_cat.mean()
    print(f'\n=== {cat_fr} ({cat_en}) — n={len(y_cat)}, winners={int(y_cat.sum())}, base-rate={base_rate:.1%} ===')

    for model_name in MODEL_PLAN[cat_en]:
        factory = MODEL_FACTORIES[model_name]
        try:
            metrics = evaluate_by_group(
                X_cat, y_cat, g_cat,
                model_factory=lambda f=factory, x=X_cat: f(x),
            )
        except Exception as e:
            print(f'  [ERR] {model_name}: {type(e).__name__}: {e}')
            metrics = {'top1_acc': float('nan'), 'top1_std': float('nan'),
                       'pr_auc': float('nan'), 'log_loss': float('nan'),
                       'n_years_evaluated': 0}

        results.append({
            'category_fr': cat_fr,
            'category': cat_en,
            'model': model_name,
            'is_theoretical_pick': model_name == THEORETICAL_PICK[cat_en],
            'n': len(y_cat),
            'base_rate': base_rate,
            **metrics,
        })
        print(f'  {model_name:18s}  top1={metrics["top1_acc"]:.3f}  '
              f'PR-AUC={metrics["pr_auc"]:.3f}  log-loss={metrics["log_loss"]:.3f}')

results_df = pd.DataFrame(results)
print(f'\n✅ Évaluation terminée : {len(results_df)} couples (catégorie × modèle).')

## 10. Synthèse — tableau croisé et comparaison vs théorie

### 10.1 Top-1 accuracy par (catégorie × modèle)

In [ ]:
pivot = (results_df
         .pivot_table(index='category_fr', columns='model', values='top1_acc')
         .reindex(index=[TARGETS[c] for c in TARGETS])
         .round(3))
pivot

### 10.2 Gagnant empirique vs pari théorique

Pour chaque catégorie : meilleur modèle observé (top-1 acc max), comparé au pari de la justification.

In [ ]:
best_per_cat = (results_df
                .sort_values('top1_acc', ascending=False)
                .groupby('category', as_index=False)
                .first())
best_per_cat['theoretical_pick'] = best_per_cat['category'].map(THEORETICAL_PICK)
best_per_cat['theoretical_top1'] = best_per_cat.apply(
    lambda r: results_df[(results_df.category == r['category']) &
                          (results_df.model == r['theoretical_pick'])]['top1_acc'].iloc[0]
    if len(results_df[(results_df.category == r['category']) &
                       (results_df.model == r['theoretical_pick'])]) else float('nan'),
    axis=1,
)
best_per_cat['theory_match'] = best_per_cat['model'] == best_per_cat['theoretical_pick']
best_per_cat['gap_vs_theory'] = (best_per_cat['top1_acc'] - best_per_cat['theoretical_top1']).round(3)

cols = ['category_fr', 'n', 'base_rate', 'model', 'top1_acc',
        'theoretical_pick', 'theoretical_top1', 'theory_match', 'gap_vs_theory']
best_per_cat[cols].sort_values('top1_acc', ascending=False).round(3)

### 10.3 Comparaison à la base-rate (sanity check)

Un modèle utile doit faire **mieux que `1/n_nominees`** (≈ base-rate). Si on est sous, le modèle est inutile.

In [ ]:
best_per_cat['lift_vs_base_rate'] = (best_per_cat['top1_acc'] - best_per_cat['base_rate']).round(3)
best_per_cat[['category_fr', 'base_rate', 'top1_acc', 'lift_vs_base_rate', 'model']]\
    .sort_values('lift_vs_base_rate', ascending=False).round(3)

## 11. Conclusion & étapes suivantes

**Ce que ce notebook a fait** :
- Implémenté un pipeline reproductible (feature engineering + évaluation `GroupKFold`) sur les 9 catégories.
- Comparé 5 modèles par catégorie (cf. justification §5–13), confrontés au pari théorique.
- Rapporté **top-1 accuracy par année** (métier), PR-AUC (déséquilibre) et log-loss (calibration).

**Limites assumées (à documenter dans la présentation)** :
1. **Petits échantillons** (100–200 lignes par catégorie) → tout écart < 3 points entre deux modèles est probablement dans le bruit de CV. Recommandation §15.9 : rapporter `moyenne ± écart-type` (déjà fait via `top1_std`).
2. **Pas de tuning d'hyperparamètres** : grilles fixées d'après les valeurs recommandées en cours. Une `GridSearchCV(cv=GroupKFold)` à grille réduite (≤ 12 combinaisons) est l'étape naturelle suivante (§15.x).
3. **Pas de features `production_company`** (animation §11.5) ni de `writer_prior_noms` (§13.5 — `writers` est multi-personnes, non explosé ici) → ces enrichissements peuvent débloquer 5–10 points sur Animation / Screenplay.
4. **Pas de calibration** des probabilités (Platt / isotonic) → indispensable si l'app Streamlit affiche `P(winner)`.
5. **Pas de PSI / drift check** entre décennies (§15.8) → à ajouter avant déploiement.
6. **Stacking minimaliste** (LR + XGBoost) : la justification §13.4 suggère une 3e branche TF-IDF+LR séparée des features numériques pour Best Screenplay. Implémentable en `FeatureUnion` à itération suivante.

**Lecture pédagogique** :
- Comparer la colonne `theory_match` du tableau §10.2 : si la majorité des paris théoriques sont confirmés, la grille de lecture du cours est solide ; sinon, identifier les catégories où la pratique a divergé et discuter *pourquoi* (taille d'échantillon, features manquantes, signal moins linéaire qu'attendu).
- Les écarts `gap_vs_theory` négatifs (théorie gagne) confortent la rigueur du framework du cours. Les écarts positifs (un autre modèle bat le pari) sont les vrais sujets de discussion.